# ESS Round 8 data curation: all-adult outputs with and without official ESS weights

This notebook constructs the ESS Round 8 respondent-level belief datasets used in the present project.

The principal outputs:

- retain respondents aged 18 or older;
- also retain respondents whose age is missing;
- **do not exclude respondents because of the number of missing constructed beliefs**;
- leave missing belief values as missing (`NaN`) rather than imputing them;
- record the number of missing and available beliefs for every respondent;
- retain a Boolean `cca_missingness_eligible` flag identifying respondents who satisfy Van Noord et al.'s historical CCA criterion of no more than two missing beliefs;
- create separate datasets with and without the four official ESS survey-weight columns.

The historical CCA-compatible subset is reconstructed only for validation against Supplementary Table A2. It is not used as the principal output sample.

## Step 1 — Locate the project, import the shared code, and define paths

Run this notebook from either the project root or the `notebooks/` folder. The following cell locates the folder containing `data/`, `notebooks/`, and `src/`, adds it to Python's import path, imports the shared curation functions, and defines the ESS8 input and output paths.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display


def locate_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if all(
            (candidate / folder).is_dir()
            for folder in ("data", "notebooks", "src")
        ):
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root containing "
        "data/, notebooks/, and src/."
    )


PROJECT_ROOT = locate_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import ess8_config as config
from src.ess_curation_common import (
    apply_analysis_sample_rule,
    construct_belief_variables,
    require_columns,
    split_weighted_and_unweighted,
    summarise_beliefs,
    valid_range,
    validate_weight_split,
)

RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / config.RAW_DATA_FOLDER
    / config.RAW_DATA_FILENAME
)
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

WITHOUT_WEIGHTS_PATH = (
    PROCESSED_DIR / config.OUTPUT_FILENAMES["without_weights"]
)
WITH_WEIGHTS_PATH = (
    PROCESSED_DIR / config.OUTPUT_FILENAMES["with_weights"]
)
VALIDATION_PATH = (
    PROCESSED_DIR / config.OUTPUT_FILENAMES["validation"]
)

print("Project root:", PROJECT_ROOT)
print("Raw ESS8 file:", RAW_DATA_PATH)
print("Processed-data folder:", PROCESSED_DIR)
print("Without-weights output:", WITHOUT_WEIGHTS_PATH.name)
print("With-weights output:", WITH_WEIGHTS_PATH.name)
print("CCA-subset validation output:", VALIDATION_PATH.name)

Project root: /Users/karan/Desktop/SSM-MERC/polarization/Github/C72H-SimPol/data_curation
Raw ESS8 file: /Users/karan/Desktop/SSM-MERC/polarization/Github/C72H-SimPol/data_curation/data/ESS8e02_3/ESS8e02_3.csv
Processed-data folder: /Users/karan/Desktop/SSM-MERC/polarization/Github/C72H-SimPol/data_curation/data/processed
Without-weights output: ess8_beliefs_all_adults_without_weights.csv
With-weights output: ess8_beliefs_all_adults_with_weights.csv
CCA-subset validation output: ess8_cca_subset_validation_against_paper.csv


## Step 2 — Load the raw ESS Round 8 file and verify the required inputs

No rows or values are changed in this step. The checks confirm the expected raw sample size, country count, required metadata variables, constituent belief items, and four official ESS weight columns.

In [2]:
raw = pd.read_csv(RAW_DATA_PATH, low_memory=False)

require_columns(
    raw,
    config.REQUIRED_RAW_COLUMNS,
    context="raw ESS Round 8 file",
)

assert len(raw) == config.EXPECTED_RAW_N, (
    f"Expected {config.EXPECTED_RAW_N:,} raw rows, "
    f"found {len(raw):,}."
)
assert (
    raw["cntry"].nunique(dropna=True)
    == config.EXPECTED_RAW_COUNTRIES
)

weight_missing_raw = (
    raw.loc[:, config.WEIGHT_COLUMNS].isna().sum()
)
assert (weight_missing_raw == 0).all(), (
    "Unexpected missing values in raw ESS weights: "
    f"{weight_missing_raw.to_dict()}"
)

print(f"Raw respondents: {len(raw):,}")
print(f"Raw columns: {raw.shape[1]:,}")
print(
    "Countries:",
    raw["cntry"].nunique(dropna=True),
)
print(
    "Official ESS weight columns:",
    list(config.WEIGHT_COLUMNS),
)
display(
    weight_missing_raw
    .rename("missing_values")
    .to_frame()
)

Raw respondents: 44,387
Raw columns: 535
Countries: 23
Official ESS weight columns: ['dweight', 'pspwght', 'pweight', 'anweight']


,missing_values
dweight,0
pspwght,0
pweight,0
anweight,0


## Step 3 — Construct identifiers, demographics, voting metadata, and official ESS weights

The metadata follow the common project schema.

- `ess_row_id` records the original row position in the raw ESS8 CSV, starting at 1.
- `ess_unique_id` combines the country code and ESS respondent ID.
- Invalid or non-substantive ESS values outside each variable's valid response range become missing.
- `education_3cat` collapses `eisced` into lower, middle, and higher education.
- `urbanization = 6 - domicil`, so higher values indicate a more urban setting.
- `vote_simple` retains the raw three-category voting-status variable.
- The four official ESS weight variables are copied directly and are not transformed.

In [3]:
metadata = pd.DataFrame(index=raw.index)

metadata["ess_row_id"] = np.arange(1, len(raw) + 1)
metadata["idno"] = pd.to_numeric(
    raw["idno"],
    errors="coerce",
).astype("Int64")
metadata["cntry"] = (
    raw["cntry"].astype("string").str.strip()
)
metadata["country_name"] = (
    metadata["cntry"].map(config.COUNTRY_LABELS)
)
metadata["ess_unique_id"] = (
    metadata["cntry"]
    + "_"
    + metadata["idno"].astype("string")
)

for column in (
    "agea",
    "gndr",
    "eisced",
    "hinctnta",
    "rlgblg",
    "blgetmg",
):
    minimum, maximum = config.METADATA_VALID_RANGES[column]
    metadata[column] = valid_range(
        raw[column],
        minimum,
        maximum,
    )

metadata["education_3cat"] = np.nan
for category, source_values in (
    config.EDUCATION_3CAT_MAP.items()
):
    metadata.loc[
        metadata["eisced"].isin(source_values),
        "education_3cat",
    ] = category

minimum, maximum = (
    config.METADATA_VALID_RANGES["domicil"]
)
domicil_clean = valid_range(
    raw["domicil"],
    minimum,
    maximum,
)
metadata["urbanization"] = (
    config.URBANIZATION_REVERSE_CONSTANT
    - domicil_clean
)

vote_minimum, vote_maximum = (
    config.METADATA_VALID_RANGES["vote"]
)
metadata["vote_simple"] = valid_range(
    raw["vote"],
    vote_minimum,
    vote_maximum,
)

for column in config.WEIGHT_COLUMNS:
    metadata[column] = pd.to_numeric(
        raw[column],
        errors="coerce",
    )

metadata = metadata.loc[
    :, config.METADATA_COLUMNS_WITH_WEIGHTS
]

assert metadata["country_name"].notna().all()
assert metadata["ess_unique_id"].notna().all()
assert metadata["ess_unique_id"].is_unique
assert (
    metadata.loc[:, config.WEIGHT_COLUMNS]
    .notna()
    .all()
    .all()
)

print("Metadata shape:", metadata.shape)
display(metadata.head())

Metadata shape: (44387, 18)


,ess_row_id,idno,cntry,country_name,ess_unique_id,agea,gndr,eisced,education_3cat,hinctnta,rlgblg,urbanization,blgetmg,vote_simple,dweight,pspwght,pweight,anweight
0,1,1,AT,Austria,AT_1,34.0,2.0,7.0,3.0,NaN,2.0,5.0,2.0,3.0,0.611677,1.178495,0.370393,0.436506
1,2,2,AT,Austria,AT_2,52.0,1.0,4.0,2.0,5.0,2.0,5.0,2.0,3.0,1.223354,0.899471,0.370393,0.333158
2,3,4,AT,Austria,AT_4,68.0,2.0,3.0,2.0,2.0,1.0,2.0,2.0,1.0,0.389058,0.315753,0.370393,0.116953
3,4,6,AT,Austria,AT_6,54.0,1.0,3.0,2.0,4.0,NaN,5.0,2.0,NaN,0.642594,0.472467,0.370393,0.174999
4,5,10,AT,Austria,AT_10,20.0,2.0,3.0,2.0,2.0,1.0,3.0,1.0,2.0,3.432402,2.246706,0.370393,0.832164


## Step 4 — Construct the 20 ESS Round 8 belief variables

The round-specific item definitions, valid response ranges, coding directions, and belief composition are stored in `src/ess8_config.py`.

Each item is cleaned and rescaled to 0–1. Multi-item beliefs use a strict row mean: all constituent items must be available, otherwise the constructed belief remains missing. No imputation is performed.

In [4]:
beliefs, coded_items = construct_belief_variables(
    raw,
    config.BELIEF_MAP,
    config.ITEM_CODING,
)

assert beliefs.shape == (
    len(raw),
    config.EXPECTED_BELIEF_COUNT,
)
assert list(beliefs.columns) == list(
    config.BELIEF_COLUMNS
)
assert list(coded_items.columns) == list(
    config.BELIEF_ITEM_COLUMNS
)

belief_minimum = beliefs.min(
    skipna=True
).min()
belief_maximum = beliefs.max(
    skipna=True
).max()

assert belief_minimum >= 0.0
assert belief_maximum <= 1.0

print(
    "Constructed belief variables:",
    beliefs.shape[1],
)
print(
    "Cleaned constituent ESS items:",
    coded_items.shape[1],
)
print(
    f"Observed belief range: "
    f"[{belief_minimum:.3f}, {belief_maximum:.3f}]"
)
display(beliefs.head())

Constructed belief variables: 20
Cleaned constituent ESS items: 38
Observed belief range: [0.000, 1.000]


,left_right_identification,gender_inequality,anti_lgbt,euroscepticism,anti_immigration,anti_egalitarianism,benefits_harm_economy,benefits_harm_society,welfare_chauvinism,anti_economic_interventionism,anti_social_benefits_low_income,anti_social_benefits_parents,educational_spending,anti_basic_income,anti_climate_change_taxes,anti_climate_change_renewables,anti_climate_ban_appliances,climate_skepticism,authoritarianism,anti_libertarianism
0,0.0,0.00,0.000000,0.5,0.000000,0.000000,0.000,0.000,0.00,0.000000,0.000000,0.333333,NaN,0.000000,0.00,0.00,0.25,0.25,0.20,0.28
1,0.1,0.25,0.000000,0.0,0.000000,0.166667,0.750,0.250,0.25,0.333333,0.666667,0.666667,0.666667,0.333333,0.00,0.25,0.25,0.25,0.24,0.44
2,0.5,0.00,0.166667,1.0,0.555556,0.666667,0.250,0.625,0.50,0.133333,0.666667,1.000000,0.333333,1.000000,0.75,0.25,0.25,0.25,0.72,0.56
3,0.0,0.50,0.416667,0.3,NaN,0.500000,0.750,0.250,0.75,0.466667,0.666667,0.333333,0.666667,NaN,0.50,0.50,0.50,0.50,0.68,0.28
4,0.5,0.00,0.000000,0.2,0.222222,0.166667,0.625,0.000,0.25,0.000000,0.333333,1.000000,0.333333,0.333333,0.50,0.00,0.75,0.25,0.48,0.40


## Step 5 — Create the principal adult analysis sample without filtering on belief missingness

The metadata and constructed beliefs are combined and ordered by country and respondent ID.

Respondents are retained when their age is at least 18 or when age is missing. **No respondent is removed because of the number of missing constructed beliefs.**

The following diagnostic columns are added:

- `n_belief_missing`: number of missing constructed beliefs;
- `n_belief_available`: number of available constructed beliefs;
- `cca_missingness_eligible`: `True` when no more than two beliefs are missing.

The last column is retained solely to reproduce the historical CCA-compatible subset when required.

In [5]:
curated_all = pd.concat(
    [metadata, beliefs],
    axis=1,
)
curated_all = curated_all.sort_values(
    ["cntry", "idno"],
    kind="stable",
).reset_index(drop=True)

analysis_sample = apply_analysis_sample_rule(
    curated_all,
    config.BELIEF_COLUMNS,
    age_column="agea",
    minimum_age=config.MINIMUM_AGE,
    cca_maximum_missing_beliefs=(
        config.CCA_MAXIMUM_MISSING_BELIEFS
    ),
).reset_index(drop=True)

assert len(analysis_sample) == config.EXPECTED_ANALYSIS_N, (
    f"Expected {config.EXPECTED_ANALYSIS_N:,} "
    f"principal-analysis rows, "
    f"found {len(analysis_sample):,}."
)
assert (
    analysis_sample["cntry"].nunique(dropna=True)
    == config.EXPECTED_ANALYSIS_COUNTRIES
)
assert (
    analysis_sample["agea"].ge(config.MINIMUM_AGE)
    | analysis_sample["agea"].isna()
).all()
assert (
    analysis_sample["n_belief_missing"]
    + analysis_sample["n_belief_available"]
).eq(config.EXPECTED_BELIEF_COUNT).all()

cca_subset = analysis_sample.loc[
    analysis_sample["cca_missingness_eligible"]
].copy()

assert len(cca_subset) == config.EXPECTED_CCA_N
assert (
    cca_subset["cntry"].nunique(dropna=True)
    == config.EXPECTED_CCA_COUNTRIES
)
assert cca_subset["n_belief_missing"].le(
    config.CCA_MAXIMUM_MISSING_BELIEFS
).all()

print(f"Raw ESS8 respondents: {len(raw):,}")
print(
    "Principal adult analysis respondents:",
    f"{len(analysis_sample):,}",
)
print(
    "Historical CCA-compatible respondents:",
    f"{len(cca_subset):,}",
)
print(
    "Adults retained despite more than two "
    "missing beliefs:",
    f"{(~analysis_sample['cca_missingness_eligible']).sum():,}",
)
print(
    "Countries retained:",
    analysis_sample["cntry"].nunique(dropna=True),
)

missingness_distribution = (
    analysis_sample["n_belief_missing"]
    .value_counts()
    .sort_index()
    .rename_axis("n_belief_missing")
    .rename("respondents")
    .to_frame()
)
missingness_distribution["percent"] = (
    100.0
    * missingness_distribution["respondents"]
    / len(analysis_sample)
)

display(missingness_distribution)

Raw ESS8 respondents: 44,387
Principal adult analysis respondents: 43,148
Historical CCA-compatible respondents: 37,118
Adults retained despite more than two missing beliefs: 6,030
Countries retained: 23


,respondents,percent
n_belief_missing,,
0,25762,59.706128
1,7634,17.692593
2,3722,8.626124
3,1971,4.567999
4,1299,3.010568
5,834,1.932882
6,528,1.223695
7,395,0.915454
8,247,0.572448


## Step 6 — Inspect country counts and belief descriptives for the principal sample

These tables describe the full adult analysis sample. They are not restricted to respondents satisfying the historical CCA missingness threshold.

In [6]:
country_counts = (
    analysis_sample
    .groupby(
        ["cntry", "country_name"],
        dropna=False,
    )
    .size()
    .rename("N")
    .reset_index()
)

analysis_belief_summary = summarise_beliefs(
    analysis_sample,
    config.BELIEF_COLUMNS,
)
analysis_belief_summary["N_total"] = len(
    analysis_sample
)
analysis_belief_summary["N_missing"] = (
    analysis_belief_summary["N_total"]
    - analysis_belief_summary["N_reproduced"]
)
analysis_belief_summary["percent_missing"] = (
    100.0
    * analysis_belief_summary["N_missing"]
    / analysis_belief_summary["N_total"]
)

print("Country-level principal sample sizes:")
display(country_counts)

print(
    "Belief-variable summary for the "
    "principal adult sample:"
)
display(analysis_belief_summary)

Country-level principal sample sizes:


,cntry,country_name,N
0,AT,Austria,1994
1,BE,Belgium,1702
2,CH,Switzerland,1465
3,CZ,Czechia,2186
4,DE,Germany,2726
5,EE,Estonia,1963
6,ES,Spain,1918
7,FI,Finland,1868
8,FR,France,2016
9,GB,United Kingdom,1925


Belief-variable summary for the principal adult sample:


,belief_variable,N_reproduced,mean_reproduced,sd_reproduced,N_total,N_missing,percent_missing
0,left_right_identification,37671,0.515556,0.224382,43148,5477,12.693520
1,gender_inequality,42672,0.241288,0.275280,43148,476,1.103180
2,anti_lgbt,40487,0.352852,0.277589,43148,2661,6.167146
3,euroscepticism,39706,0.512457,0.268111,43148,3442,7.977195
4,anti_immigration,41173,0.463462,0.269762,43148,1975,4.577269
5,anti_egalitarianism,41535,0.377144,0.191835,43148,1613,3.738296
6,benefits_harm_economy,39379,0.488208,0.226977,43148,3769,8.735051
7,benefits_harm_society,41349,0.414747,0.220312,43148,1799,4.169371
8,welfare_chauvinism,41216,0.549028,0.261911,43148,1932,4.477612
9,anti_economic_interventionism,42218,0.241808,0.162478,43148,930,2.155372


## Step 7 — Validate the historical CCA-compatible subset against Supplementary Table A2

Supplementary Table A2 was reproduced using the historical sample restriction of no more than two missing beliefs. Therefore, the published counts, means, and standard deviations must be compared with `cca_subset`, not with the new principal all-adult dataset.

This validation does not remove respondents from the saved principal outputs.

In [7]:
cca_reproduced = summarise_beliefs(
    cca_subset,
    config.BELIEF_COLUMNS,
)

paper_statistics = pd.DataFrame(
    config.PAPER_STATISTICS,
    columns=config.PAPER_STATISTICS_COLUMNS,
)

validation = paper_statistics.merge(
    cca_reproduced,
    on="belief_variable",
    how="left",
    validate="one_to_one",
)

validation["mean_reproduced_round2"] = (
    validation["mean_reproduced"].round(2)
)
validation["sd_reproduced_round2"] = (
    validation["sd_reproduced"].round(2)
)
validation["N_matches"] = (
    validation["N_paper"]
    == validation["N_reproduced"]
)
validation["mean_matches_round2"] = (
    validation["mean_paper"]
    == validation["mean_reproduced_round2"]
)
validation["sd_matches_round2"] = (
    validation["sd_paper"]
    == validation["sd_reproduced_round2"]
)

match_columns = [
    "N_matches",
    "mean_matches_round2",
    "sd_matches_round2",
]
assert validation.loc[:, match_columns].all().all()

print(
    "Historical CCA subset:",
    f"{len(cca_subset):,} respondents",
)
print(
    "All 20 beliefs reproduce "
    "Supplementary Table A2."
)
display(validation)

Historical CCA subset: 37,118 respondents
All 20 beliefs reproduce Supplementary Table A2.


,belief_variable,N_paper,mean_paper,sd_paper,N_reproduced,mean_reproduced,sd_reproduced,mean_reproduced_round2,sd_reproduced_round2,N_matches,mean_matches_round2,sd_matches_round2
0,left_right_identification,34248,0.51,0.22,34248,0.512920,0.223370,0.51,0.22,True,True,True
1,gender_inequality,37038,0.23,0.27,37038,0.227955,0.269111,0.23,0.27,True,True,True
2,anti_lgbt,36098,0.34,0.27,36098,0.336637,0.270381,0.34,0.27,True,True,True
3,euroscepticism,35848,0.51,0.27,35848,0.508968,0.266828,0.51,0.27,True,True,True
4,anti_immigration,36459,0.45,0.27,36459,0.451734,0.265234,0.45,0.27,True,True,True
5,anti_egalitarianism,36772,0.38,0.19,36772,0.380239,0.193550,0.38,0.19,True,True,True
6,benefits_harm_economy,35956,0.49,0.23,35956,0.490534,0.226702,0.49,0.23,True,True,True
7,benefits_harm_society,36801,0.41,0.22,36801,0.410132,0.218079,0.41,0.22,True,True,True
8,welfare_chauvinism,36441,0.54,0.26,36441,0.543090,0.258760,0.54,0.26,True,True,True
9,anti_economic_interventionism,36927,0.24,0.16,36927,0.244343,0.159874,0.24,0.16,True,True,True


## Step 8 — Create separate principal datasets with and without official weights

The two outputs contain exactly the same respondents, identifiers, demographics, belief values, missingness counts, and CCA-eligibility flags.

The with-weights version additionally includes `dweight`, `pspwght`, `pweight`, and `anweight` immediately after `vote_simple`. No rows are duplicated and no belief values are multiplied by weights.

In [8]:
analysis_without_weights, analysis_with_weights = (
    split_weighted_and_unweighted(
        analysis_sample,
        config.WEIGHT_COLUMNS,
        insert_after=config.WEIGHT_INSERT_AFTER,
    )
)

analysis_without_weights = (
    analysis_without_weights.loc[
        :, config.OUTPUT_COLUMNS_WITHOUT_WEIGHTS
    ].copy()
)
analysis_with_weights = (
    analysis_with_weights.loc[
        :, config.OUTPUT_COLUMNS_WITH_WEIGHTS
    ].copy()
)

assert (
    analysis_without_weights.shape
    == config.EXPECTED_WITHOUT_WEIGHTS_SHAPE
)
assert (
    analysis_with_weights.shape
    == config.EXPECTED_WITH_WEIGHTS_SHAPE
)

validate_weight_split(
    analysis_without_weights,
    analysis_with_weights,
    config.WEIGHT_COLUMNS,
    require_complete_weights=True,
)

print(
    "Without-weights shape:",
    analysis_without_weights.shape,
)
print(
    "With-weights shape:",
    analysis_with_weights.shape,
)
print(
    "The datasets differ only by the four "
    "official weight columns."
)

Without-weights shape: (43148, 37)
With-weights shape: (43148, 41)
The datasets differ only by the four official weight columns.


## Step 9 — Save the principal datasets and historical validation table

The principal all-adult outputs and the separate historical CCA-subset validation table are written to `data/processed/` using the filenames defined in `src/ess8_config.py`.

The old `ess8_cca_initial_beliefs_*.csv` files are not overwritten automatically.

In [9]:
analysis_without_weights.to_csv(
    WITHOUT_WEIGHTS_PATH,
    index=False,
)
analysis_with_weights.to_csv(
    WITH_WEIGHTS_PATH,
    index=False,
)
validation.to_csv(
    VALIDATION_PATH,
    index=False,
)

print("Saved:")
print(" -", WITHOUT_WEIGHTS_PATH)
print(" -", WITH_WEIGHTS_PATH)
print(" -", VALIDATION_PATH)

Saved:
 - /Users/karan/Desktop/SSM-MERC/polarization/Github/C72H-SimPol/data_curation/data/processed/ess8_beliefs_all_adults_without_weights.csv
 - /Users/karan/Desktop/SSM-MERC/polarization/Github/C72H-SimPol/data_curation/data/processed/ess8_beliefs_all_adults_with_weights.csv
 - /Users/karan/Desktop/SSM-MERC/polarization/Github/C72H-SimPol/data_curation/data/processed/ess8_cca_subset_validation_against_paper.csv


## Step 10 — Read the saved files back and perform final integrity checks

Reading the outputs back from disk detects problems involving serialization, accidental index columns, schema order, respondent identity, missingness indicators, or survey weights.

In [10]:
saved_without = pd.read_csv(
    WITHOUT_WEIGHTS_PATH,
    low_memory=False,
)
saved_with = pd.read_csv(
    WITH_WEIGHTS_PATH,
    low_memory=False,
)

assert (
    saved_without.shape
    == config.EXPECTED_WITHOUT_WEIGHTS_SHAPE
)
assert (
    saved_with.shape
    == config.EXPECTED_WITH_WEIGHTS_SHAPE
)
assert list(saved_without.columns) == list(
    config.OUTPUT_COLUMNS_WITHOUT_WEIGHTS
)
assert list(saved_with.columns) == list(
    config.OUTPUT_COLUMNS_WITH_WEIGHTS
)

validate_weight_split(
    saved_without,
    saved_with,
    config.WEIGHT_COLUMNS,
    require_complete_weights=True,
)

assert saved_without["ess_unique_id"].is_unique
assert saved_with["ess_unique_id"].is_unique
assert saved_without["ess_unique_id"].equals(
    saved_with["ess_unique_id"]
)
assert (
    saved_without["n_belief_missing"]
    + saved_without["n_belief_available"]
).eq(config.EXPECTED_BELIEF_COUNT).all()

saved_cca_eligible = (
    saved_without["cca_missingness_eligible"]
    .astype("boolean")
)
assert int(saved_cca_eligible.sum()) == config.EXPECTED_CCA_N

print("ESS Round 8 curation completed successfully.")
print(f"Raw respondents: {config.EXPECTED_RAW_N:,}")
print(
    "Principal adult respondents:",
    f"{len(saved_without):,}",
)
print(
    "Historical CCA-compatible respondents:",
    f"{int(saved_cca_eligible.sum()):,}",
)
print(
    "Adults retained with more than two "
    "missing beliefs:",
    f"{int((~saved_cca_eligible).sum()):,}",
)
print(
    "Countries:",
    saved_without["cntry"].nunique(dropna=True),
)
print(
    "Belief variables:",
    len(config.BELIEF_COLUMNS),
)
print(
    "Without-weights columns:",
    saved_without.shape[1],
)
print(
    "With-weights columns:",
    saved_with.shape[1],
)
print("Missing official weights:")
print(
    saved_with.loc[:, config.WEIGHT_COLUMNS]
    .isna()
    .sum()
    .to_dict()
)

ESS Round 8 curation completed successfully.
Raw respondents: 44,387
Principal adult respondents: 43,148
Historical CCA-compatible respondents: 37,118
Adults retained with more than two missing beliefs: 6,030
Countries: 23
Belief variables: 20
Without-weights columns: 37
With-weights columns: 41
Missing official weights:
{'dweight': 0, 'pspwght': 0, 'pweight': 0, 'anweight': 0}
